In [19]:
using Random

using GaussianMixtures: GMM
using GaussianProcesses: GPE, MeanZero, SE
using ScikitLearn
using ScikitLearn.CrossValidation: train_test_split

@sk_import linear_model: LogisticRegression

Random.seed!(12);

┌ Info: Precompiling GaussianProcesses [891a1506-143c-57d2-908e-e1f8e92e6de9]
└ @ Base loading.jl:1260


In [9]:
X = rand(20, 3)
y = rand([true, false], 20);

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=11)
logreg = fit!(LogisticRegression(penalty="l2"), X_train, y_train)
preds = predict(logreg, X_test)

5-element Array{Bool,1}:
 0
 1
 1
 1
 1

In [12]:
y_test

5-element Array{Bool,1}:
 1
 0
 1
 1
 1

In [16]:
gmm = fit!(GMM(n_components=3, kind=:diag), X_train)

┌ Info: Initializing GMM, 3 Gaussians diag covariance 3 dimensions using 15 data points
└ @ GaussianMixtures /Users/dsatterthwaite/.julia/packages/GaussianMixtures/3jRIL/src/train.jl:78


  Iters               objv        objv-change | affected 
-------------------------------------------------------------
      0       2.680731e+00
      1       1.583045e+00      -1.097686e+00 |        0
      2       1.583045e+00       0.000000e+00 |        0
K-means converged with 2 iterations (objv = 1.5830454705027663)


┌ Info: K-means with 15 data points using 2 iterations
│ 1.3 data points per parameter
└ @ GaussianMixtures /Users/dsatterthwaite/.julia/packages/GaussianMixtures/3jRIL/src/train.jl:139


GMM{Float64} with 3 components in 3 dimensions and diag covariance
⋮


In [20]:
gp = fit!(GPE(; mean=MeanZero(), kernel=SE(0., 0.), logNoise=-1e8), 
          X_train, 
          Float64.(y_train))

GP Exact object:
  Dim = 3
  Number of observations = 15
  Mean function:
    Type: MeanZero, Params: Float64[]
  Kernel:
    Type: GaussianProcesses.SEIso{Float64}, Params: [0.0, 0.0]
  Input observations = 
[0.8462559286657476 0.4939533331843271 … 0.7226206182012778 0.7620498237481113; 0.4022085633806616 0.35584387101099035 … 0.9634201804282001 0.184913946607548; 0.3539278221243245 0.8483691343973072 … 0.196294750679592 0.5915047929117252]
  Output observations = [1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 1.0]
  Variance of observation noise = 0.0
  Marginal Log-Likelihood = -170.197

In [21]:
predict(gp, X_test)

5-element Array{Float64,1}:
 -0.08789649260620536
  0.327392621143872
  1.2907810289314057
  0.7517849571690931
  0.8401698492312448

In [22]:
y_test

5-element Array{Bool,1}:
 1
 0
 1
 1
 1